In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [3]:
# Load the training and development datasets
train_df = pd.read_csv("data/Data_RSW.csv")
dev_df = pd.read_csv("data/development_data.csv")
sc = StandardScaler()

target_column = "PullTest (N)" 

x_train, y_train = get_features_and_target(train_df, target_column)
x_dev, y_dev = get_features_and_target(dev_df, target_column)

x_train_scale = sc.fit_transform(X=x_train)
x_dev_scale = sc.transform(x_dev)


In [4]:
train_df.shape[0]

4186

# Defining Model

In [5]:
#model = 'XGBoost'
model = 'TabPFNRegressor'

# Fit Model

In [6]:
if model == 'TabPFNRegressor':

    # Initialize the regressor
    regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
    # To use TabPFN v2:
    # regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
    regressor.fit(x_train_scale, y_train)

    # Predict on the test set
    predictions = regressor.predict(x_dev_scale)

elif model == 'XGBoost':

    # Convert the data into DMatrix format
    dtrain = xgb.DMatrix(x_train_scale, label=y_train)
    dtest = xgb.DMatrix(x_dev_scale, label=y_dev)

    # Set the parameters for the XGBoost model
    params = {
        'objective': 'reg:squarederror',
        'max_depth': 5,
        'eta': 0.3,
        'eval_metric': 'rmse'
    }

    # Train the model
    num_boost_round = 10
    bst = xgb.train(params, dtrain, num_boost_round)

    # Make predictions
    predictions = bst.predict(dtest)


In [7]:
predictions

array([4090.9836, 1944.4126, 2341.2375, 2796.076 , 5243.088 , 2750.7375,
       3256.2112, 3384.3347, 3244.812 , 3228.503 , 3003.1094, 2945.0989,
       2931.1726, 2877.374 , 3049.6848, 2976.39  , 2781.2217, 2844.2034,
       2728.2366, 2644.2568, 2583.845 , 2996.4915, 2785.0198, 2656.291 ,
       2777.8296, 2650.59  , 2959.569 , 2832.6782, 2671.401 , 2706.759 ,
       2666.0554, 2800.7485, 2778.7542, 2767.2158, 3054.8965, 2607.1875,
       2747.8484, 2830.0962, 2650.2847, 2727.0317, 2679.0164, 3226.8833,
       3216.8264, 3039.1658, 3013.3684, 3057.1868, 3224.9065, 3092.5728,
       2920.4912, 3009.2207, 3156.56  , 3193.008 , 3129.328 , 3151.6099,
       3015.0806, 3168.4172, 3016.7263, 3060.846 , 3012.4546, 3101.6567,
       3036.0876, 2947.7852, 2835.7307, 2898.5706, 2614.2869, 3068.6467,
       2840.16  , 3040.7942, 2975.3137, 3054.4927, 3141.5   , 3110.5344,
       2612.1316, 2960.239 , 2764.5386, 3081.3987, 3115.247 , 2905.8967,
       2794.1318, 2954.0159, 2815.7893, 3251.2825, 

# Check Validation Data

In [8]:
# Category array (must be aligned with y_dev)
categories = dev_df.groupby("Sample ID")["Category"].first().values

plot_visualizer(
    true_vals=y_dev,
    pred_vals=predictions,
    categories=categories,
    title=f"Validation Samples: True vs Prediction (TabPFN) by Category"
)

# Check Validation Loss and R2

In [9]:

# Calculate MAE and RMSE and R2
mae  = mean_absolute_error(y_dev, predictions)
rmse = np.sqrt(root_mean_squared_error(y_dev, predictions))**2
R2   = r2_score(y_dev, predictions)



print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2: {R2:.2f}")


MAE:  7.53
RMSE: 25.69
R2: 0.99


# Maybe Feature Importance?


In [10]:
#feature_names = x_train.columns.tolist()
# Fix: tell SHAP what the model's feature names are 
#regressor.feature_names_in_ = np.array(feature_names)

# Calculate SHAP values
#shap_values = interpretability.shap.get_shap_values(
#    estimator=regressor,
#    test_x=test_x,
#    attribute_names=feature_names,
#    algorithm="permutation",
#)

# Create visualization
#fig = interpretability.shap.plot_shap(shap_values)

In [11]:
#x_dev.columns


# Cross Validation

In [12]:
cross_df = pd.read_csv("data/train_dev_data.csv")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mae_list = []
rmse_list = []
R2_list = []

for fold, (train_index, val_index) in enumerate(skf.split(cross_df["Sample ID"], cross_df["Category"])):
    x_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column) 
    x_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

    x_tr_scale = sc.fit_transform(X=x_tr)
    x_val_scale = sc.transform(x_val)

    if model == 'XGBoost':
        dtrain = xgb.DMatrix(x_tr_scale, label=y_tr)
        dval = xgb.DMatrix(x_val_scale, label=y_val)

        params = {
            'objective': 'reg:squarederror',
            'max_depth': 5,
            'eta': 0.3,
            'eval_metric': 'rmse'
        }

        num_boost_round = 5
        bst = xgb.train(params, dtrain, num_boost_round)

        preds = bst.predict(dval)
        
    else:
        regressor = TabPFNRegressor()
        regressor.fit(x_tr, y_tr)

        preds = regressor.predict(x_val)

    mae  = mean_absolute_error(y_val, preds)
    rmse = root_mean_squared_error(y_val, preds)
    R2   = r2_score(y_val, preds)

    mae_list.append(mae) 
    rmse_list.append(rmse) 
    R2_list.append(R2)

    # Categories
    categories= cross_df.iloc[val_index]["Category"].values

    plot_visualizer(
        true_vals=y_val,
        pred_vals=preds,
        categories=categories,
        title=f"Fold {fold+1}: True vs Prediction (TabPFN) by Category"
    )

    print(f"\nFold {fold+1}")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R²  :", R2)

mae_mean = np.mean(mae_list) 
rmse_mean = np.mean(rmse_list) 
R2_mean = np.mean(R2_list) 



Fold 1
MAE : 121.69049105257602
RMSE: 186.81721187084392
R²  : 0.6340417995056008



Fold 2
MAE : 138.77436919341218
RMSE: 283.44597773148706
R²  : 0.7014452213348172



Fold 3
MAE : 127.33780352618241
RMSE: 209.66846719006563
R²  : 0.78451618691722



Fold 4
MAE : 111.85529521220441
RMSE: 149.54616327938228
R²  : 0.7122334393274238



Fold 5
MAE : 107.17089776862157
RMSE: 173.63941221550584
R²  : 0.6610089922659921


In [13]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")

mean MAE:  121.37
mean RMSE: 200.62
mean R²:   0.70


In [14]:
print(f"mean MAE:  {mae_mean:.2f}")
print(f"mean RMSE: {rmse_mean:.2f}")
print(f"mean R²:   {R2_mean:.2f}")


mean MAE:  121.37
mean RMSE: 200.62
mean R²:   0.70


# Select between XGB and RandomForest

In [22]:
model_type = "xgbregressor"
#model_type = "random_forest" 

# Model Creation

In [23]:
def create_model(model_type, params):
    if model_type == "xgbregressor":
        return XGBRegressor(
            objective='reg:squarederror',
            max_depth=params['max_depth'],
            learning_rate=params['learning_rate'],
            n_estimators=params['n_estimators'],
            eval_metric='rmse'
        )
    elif model_type == "random_forest":
        return RandomForestRegressor(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            min_samples_split=params['min_samples_split'],
            min_samples_leaf=params['min_samples_leaf'],
            random_state=42,
            n_jobs=-1
        )


# Parameter Selection

In [31]:
if model_type == "xgbregressor":
    param_grid = []
    max_depth_values = range(1, 51)
    eta_values = [0.2, 0.25, 0.3, 0.35, 0.4]
    n_estimators_values = range(1, 51)

    for md in max_depth_values:
        for eta in eta_values:
            for ne in n_estimators_values:
                param_grid.append({
                    'max_depth': md,
                    'learning_rate': eta,
                    'n_estimators': ne
                })

elif model_type == "random_forest":
    param_grid = []
    n_estimators_values = range(9, 14)
    max_depth_values = range(4, 7)
    min_samples_split_values = [4,5,6,7,8,9]
    min_samples_leaf_values = range(3, 6)

    for ne in n_estimators_values:
        for md in max_depth_values:
            for mss in min_samples_split_values:
                for msl in min_samples_leaf_values:
                    param_grid.append({
                        'n_estimators': ne,
                        'max_depth': md,
                        'min_samples_split': mss,
                        'min_samples_leaf': msl
                    })


# Gridsearch

In [32]:
best_rmse = float("inf")
best_params = None
best_model = None

for params in param_grid:
    print(f"\nTesting params: {params}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    mae_list = []
    rmse_list = []
    R2_list = []

    for fold, (train_index, val_index) in enumerate(
            skf.split(cross_df["Sample ID"], cross_df["Category"])):

        X_tr, y_tr = get_features_and_target(cross_df.iloc[train_index], target_column)
        X_val, y_val = get_features_and_target(cross_df.iloc[val_index], target_column)

        X_tr_scale = sc.fit_transform(X_tr)
        X_val_scale = sc.transform(X_val)

        model = create_model(model_type, params)
        
        model.fit(X_tr_scale, y_tr)
        preds = model.predict(X_val_scale)

        mae = mean_absolute_error(y_val, preds) 
        rmse = root_mean_squared_error(y_val, preds) 
        R2 = r2_score(y_val, preds) 

        mae_list.append(mae) 
        rmse_list.append(rmse) 
        R2_list.append(R2)

    mae_mean = np.mean(mae_list) 
    rmse_mean = np.mean(rmse_list) 
    R2_mean = np.mean(R2_list) 

    if rmse_mean < best_rmse:
        best_rmse = rmse_mean
        best_params = params
        best_model = model

        print("\n==============================")
        print(" BEST MODEL FOUND ")
        print("==============================")
        print(f"Best RMSE:   {best_rmse:.2f}")
        print(f"Best Params: {best_params}")



Testing params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 1}

 BEST MODEL FOUND 
Best RMSE:   342.98
Best Params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 1}

Testing params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 2}

 BEST MODEL FOUND 
Best RMSE:   323.01
Best Params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 2}

Testing params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 3}

 BEST MODEL FOUND 
Best RMSE:   311.96
Best Params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 3}

Testing params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 4}

 BEST MODEL FOUND 
Best RMSE:   304.06
Best Params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 4}

Testing params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 5}

 BEST MODEL FOUND 
Best RMSE:   294.54
Best Params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators': 5}

Testing params: {'max_depth': 1, 'learning_rate': 0.2, 'n_estimators'

In [30]:
print("\n==============================")
print(" BEST MODEL FOUND ")
print("==============================")
print(f"Best RMSE:   {best_rmse:.2f}")
print(f"Best Params: {best_params}")

#==============================
# BEST MODEL FOUND XGBoost
#==============================
#Best RMSE:   245.17
#Best Params: {'max_depth': 5, 'learning_rate': 0.3, 'n_estimators': 5}

#==============================
# BEST MODEL FOUND RandomForest
#==============================
#Best RMSE:   208.64
#Best Params: {'n_estimators': 11, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 4}


 BEST MODEL FOUND 
Best RMSE:   232.22
Best Params: {'max_depth': 1, 'learning_rate': 0.4, 'n_estimators': 9}
